In [ ]:
import matplotlib as mpl
import matplotlib.ticker as mticker


mpl.rcParams.update({

    # -------- Lines --------
    "lines.linewidth": 2,
    "lines.markersize": 4.5,

    # -------- Axes labels --------
    "axes.labelsize": 14,
    "axes.titlesize": 14,

    # -------- Tick labels --------
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    # -------- Legend --------
    "legend.fontsize": 11,

    # -------- Grid --------
    "grid.alpha": 0.25,

})

plot A Function

In [ ]:
from pathlib import Path

def plot_A(ax,
           base_dir=Path("/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
                         "mut_project_updates/figures/circ_model/plot_a/plot_a_files"),
           phenotype_length=50,
           bin_width=3.0):
    """
    Circadian — Plot A (self-contained).
    """

    import numpy as np
    import pandas as pd
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    # ---------- helpers ----------
    def load_dataset(file_path, name):
        print(f"\nLoading {name} from: {file_path}")
        file_path = Path(file_path)
        if not file_path.exists():
            raise FileNotFoundError(f"{file_path} not found")

        try:
            df = pd.read_csv(file_path, sep="\t")
        except Exception:
            df = pd.read_csv(file_path, sep="\t", engine="python")

        if df.columns.size and isinstance(df.columns[0], str) and df.columns[0].startswith("#"):
            df = pd.read_csv(
                file_path,
                sep="\t",
                skiprows=1,
                names=["file_number", "genotype_raw", "phenotype_binary", "complexity_entropy"],
                engine="python"
            )

        if "phenotype_binary" not in df.columns and "phenotype" in df.columns:
            df = df.rename(columns={"phenotype": "phenotype_binary"})

        if "phenotype_binary" not in df.columns:
            raise ValueError(f"'phenotype_binary' column not found in {file_path}")

        df["phenotype_binary"] = df["phenotype_binary"].astype(str)
        df["phenotype_length"] = df["phenotype_binary"].str.len()
        df_filtered = df[df["phenotype_length"] == phenotype_length]

        if "complexity_entropy" not in df_filtered.columns:
            possible = [c for c in df_filtered.columns if "complex" in c.lower()]
            if possible:
                df_filtered = df_filtered.rename(columns={possible[0]: "complexity_entropy"})
            else:
                raise ValueError(f"No complexity column found in {file_path}")

        complexities = pd.to_numeric(
            df_filtered["complexity_entropy"], errors="coerce"
        ).dropna().to_numpy()

        complexities = complexities[complexities > 0]

        return complexities

    # ---------- load data ----------
    rand_file = Path(base_dir) / "t0to50_table.txt"
    zero_file = Path(base_dir) / "compiled_0mut.txt"

    rand_raw = load_dataset(rand_file, "Random")
    zero_raw = load_dataset(zero_file, "Whole_space")

    g_complexity_mean = np.mean(rand_raw)
    p_complexity_mean = np.mean(zero_raw)
    print(f"G-dist mean complexity: {g_complexity_mean:.6f}")
    print(f"P-dist mean complexity: {p_complexity_mean:.6f}")
    print(f"Number of samples in G-dist: {len(rand_raw)}")
    print(f"Number of samples in P-dist: {len(zero_raw)}")

    # ---------- compute range and bins ----------
    all_data = np.concatenate([rand_raw, zero_raw])
    min_val = float(np.min(all_data))
    max_val = float(np.max(all_data))

    bins = np.arange(min_val, max_val + bin_width, bin_width)
    n_bins = len(bins) - 1

    # ---------- histograms ----------
    rand_counts, bin_edges = np.histogram(rand_raw, bins=bins, density=False)
    zero_counts, _ = np.histogram(zero_raw, bins=bins, density=False)

    if rand_counts.sum() > 0:
        rand_counts = rand_counts / rand_counts.sum()
    if zero_counts.sum() > 0:
        zero_counts = zero_counts / zero_counts.sum()

    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2.0

    # ---------- adaptive epsilon ----------
    nonzeros = np.concatenate([
        rand_counts[rand_counts > 0],
        zero_counts[zero_counts > 0]
    ])

    eps = float(np.min(nonzeros) * 0.1) if nonzeros.size else 1e-12

    rand_safe = np.clip(rand_counts, eps, None)
    zero_safe = np.clip(zero_counts, eps, None)

    # ---------- main plot ----------
    ax.plot(bin_centers, rand_counts,
            marker="o",
            color="orange",
            label="G-dist")

    ax.fill_between(bin_centers, rand_counts, 0,
                    color="orange",
                    alpha=0.18)

    ax.plot(bin_centers, zero_counts,
            marker="o",
            color="blue",
            label="P-dist")

    ax.fill_between(bin_centers, zero_counts, 0,
                    color="blue",
                    alpha=0.15)

    ax.set_xlabel("Complexity (Lempel-Ziv)")
    ax.set_ylabel("Relative Frequency")

    ax.legend(frameon=False, loc="upper left")

    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.set_axisbelow(True)

    # ---------- inset ----------
    ax_inset = inset_axes(ax, width="22%", height="22%", loc="upper right")

    ax_inset.plot(bin_centers, rand_safe, color="orange")
    ax_inset.plot(bin_centers, zero_safe, color="blue")

    ax_inset.set_yscale("log")
    ax_inset.set_xlabel("Complexity", fontsize=9)
    ax_inset.set_ylabel("Log Frequency", fontsize=9)
    ax_inset.tick_params(labelsize=8)

    return dict(bin_width=bin_width, n_bins=n_bins, epsilon=eps)

plot B function

In [ ]:
def plot_B(ax):
    """
    Circadian — Plot B
    Entropy vs scaled mean complexity comparison.
    """

    import pandas as pd
    from pathlib import Path

    # --------------------------------------------------
    # Load NEW circadian results (with cluster points)
    # --------------------------------------------------

    new_file = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/circ_model/"
        "plot_a/plot_a_files/plotA_circadian_with_cluster_points.txt"
    )

    df_new = pd.read_csv(new_file, sep=r"\s+")

    N_new = df_new["sample_size"].values
    scaled_global_new = df_new["scaled_global"].values


    # --------------------------------------------------
    # Load OLD Pascal data
    # --------------------------------------------------

    base_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/circ_model/"
        "plot_b/plot_b_files"
    )

    dir_mapping = {
        "10": 10,
        "100": 100,
        "103": 1000,
        "104": 10000,
        "105": 100000,
        "106": 1000000
    }

    old_data = []

    for dir_name, N in dir_mapping.items():
        dir_path = base_dir / dir_name
        
        entropy_file = dir_path / "entropy" / f"dataset{dir_name}_entropy_value.txt"
        scaled_mean_file = dir_path / "lz_values" / "mean" / "scaled_mean.txt"
        
        try:
            with open(entropy_file, 'r') as f:
                entropy_old = float(f.read().strip())
            
            with open(scaled_mean_file, 'r') as f:
                scaled_old = float(f.read().strip())
            
            old_data.append((N, entropy_old, scaled_old))
            
        except Exception as e:
            print(f"Warning: Could not read data for N={N}: {e}")

    old_data.sort(key=lambda x: x[0])

    N_old = [d[0] for d in old_data]
    entropy_old = [d[1] for d in old_data]
    scaled_old = [d[2] for d in old_data]


    # --------------------------------------------------
    # Plot comparison
    # --------------------------------------------------

    # Pascal entropy (green)
    ax.semilogx(
        N_old, entropy_old,
        'o-',
        color='green',
        label=r'$S$'
    )

    # New global scaling including cluster data (orange)
    ax.semilogx(
        N_new, scaled_global_new,
        'o-',
        color='orange',
        label=r'$\langle \tilde K_g \rangle$'
    )

    ax.set_xlabel('Number of sampled genotypes (N, log scale)')
    ax.set_ylabel(r'Entropy (S) and scaled $\langle \tilde{K}_{g} \rangle$')

    ax.set_xscale("log")
    ax.minorticks_off()
    ax.set_xlim(5, 2000000)
    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    ax.legend(frameon=False)

Plot C function

In [ ]:
def plot_C(ax):
    """
    Circadian — Plot C
    Mean complexity vs Shannon entropy (linear regression).
    """

    import numpy as np
    from pathlib import Path

    base_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/"
        "circadian/mut_project_updates/figures/circ_model/"
        "plot_c/plot_c_files/plot_d"
    )

    min_L, max_L = 15, 50

    def read_float(path):
        return float(path.read_text().strip())

    entropy_vals = []
    simple_means = []
    labels = []

    for L_dir in sorted(base_dir.iterdir(), key=lambda p: int(p.name) if p.name.isdigit() else -1):
        if not (L_dir.is_dir() and L_dir.name.isdigit()):
            continue

        L = int(L_dir.name)
        if not (min_L <= L <= max_L):
            continue

        try:
            entropy = read_float(L_dir / "entropy" / f"dataset{L}_entropy_value.txt")
            simple = read_float(L_dir / "lz_values/mean/simple_mean.txt")

            entropy_vals.append(entropy)
            simple_means.append(simple)
            labels.append(str(L))

        except:
            continue

    entropy_vals = np.array(entropy_vals)
    simple_means = np.array(simple_means)

    # Linear regression
    m, b = np.polyfit(entropy_vals, simple_means, 1)
    fit_line = m * entropy_vals + b
    sign = "-" if b < 0 else "+"
    eq_label = rf"$\langle \tilde{{K}}_{{g}} \rangle = {m:.2f}\,S {sign} {abs(b):.2f}$"

    ax.scatter(entropy_vals, simple_means, color='orange')
    ax.plot(entropy_vals, fit_line, color='orange', label=eq_label)

    for xi, yi, lbl in zip(entropy_vals, simple_means, labels):
        ax.text(xi, yi, lbl, fontsize=11, ha='center', va='bottom')

    ax.set_xlabel("Shannon Entropy (S)")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")
    #ax.grid(True, which="major", linestyle="-", alpha=0.2)
    ax.margins(x=0.05, y=0.05)
    ax.legend(frameon=False)

plot D function

In [ ]:
def plot_D(ax):
    """
    Circadian — Plot D
    Mean LZ vs mutation number (no error bars).
    """

    import numpy as np
    import pandas as pd
    from pathlib import Path
    import matplotlib.ticker as mticker

    # ---------- configure ----------
    tables_dir = Path(
        "/Users/sam/Documents/Oxford/Physics/sloppiness/circadian/"
        "mut_project_updates/figures/circ_model/plot_d/plot_d_files/tables"
    )

    groups = ["G1", "G2", "G3", "G4", "G5"]
    max_generation = 20

    marker = "o"
    # --------------------------------

    summary_stats = {}

    for group in groups:
        generations = []
        means = []

        for gen in range(0, max_generation + 1):
            table_file = tables_dir / f"{group}_{gen}param_table.txt"

            if table_file.exists():
                df = pd.read_csv(table_file, sep="\t")
                lz_vals = pd.to_numeric(
                    df["complexity_entropy"], errors="coerce"
                ).dropna().values

                if len(lz_vals) > 0:
                    generations.append(gen)
                    means.append(np.mean(lz_vals))

        summary_stats[group] = {
            "generations": np.array(generations),
            "means": np.array(means),
        }

    # --------------------------------
    # Plot
    # --------------------------------

    palette = [
        "#E8B5D6",  # G1
        "#CC79A7",  # G2
        "#A63478",  # G3
        "#7A1F5C",  # G4
        "#4A0D3A",  # G5
    ]

    for i, group in enumerate(groups):
        data = summary_stats[group]
        if len(data["generations"]) == 0:
            continue

        ax.plot(
            data["generations"],
            data["means"],
            marker=marker,
            color=palette[i],
            label=group
        )

    ax.set_xlabel("Number of mutations")
    ax.set_ylabel(r"Mean Complexity $\langle \tilde{K}_{g} \rangle$")

    ax.grid(True, which="major", linestyle="-", alpha=0.2)

    # Reverse legend order (G5 → G1)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[::-1], labels[::-1],
              loc="upper right",
              frameon=False)

    # 0.5 tick spacing margin
    all_gens = sorted(
        {g for group in summary_stats.values()
         for g in group["generations"]}
    )

    if len(all_gens) > 1:
        spacing = all_gens[1] - all_gens[0]
    else:
        spacing = 1.0

    ax.set_xlim(min(all_gens) - 0.5 * spacing,
                max(all_gens) + 0.5 * spacing)

    # Force integer ticks every 4 units
    ax.xaxis.set_major_locator(mticker.MultipleLocator(4))
    ax.xaxis.set_minor_locator(mticker.NullLocator())

create master figure

In [ ]:
import matplotlib.pyplot as plt

# ===================================
# Create master 2x2 figure
# ===================================
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes = axes.flatten()

# Call your plotting functions
plot_A(axes[0])
plot_B(axes[1])
plot_C(axes[2])
plot_D(axes[3])

# ===================================
# Panel labels underneath (standardized)
# ===================================
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for ax, label in zip(axes, panel_labels):
    ax.text(
        0.5,
        -0.14,
        label,
        transform=ax.transAxes,
        ha='center',
        va='top',
        fontsize=11
    )

plt.tight_layout()
plt.subplots_adjust(hspace=0.28, wspace=0.25)

plt.show()